# Real Out-of-Sample Validation for the Other Selectable Momentum Strategies, Weekly Regime

Epic 18: extends Epic 17's strategy-type walk-forward + holdout methodology
(`notebooks/research/out_of_sample_validation_strategy_types.ipynb`) to the **weekly** regime,
mirroring Epic 16's own monthly-to-weekly extension of Epic 15. Epic 17 only validated
`portfolio1`'s monthly regime; `portfolio2`/`portfolio3` both run this exact weekly regime
(`holding_period: 0.25`, `lookback_period: 1.0`) today.

**Two real mechanical facts confirmed by reading the code directly before running this, not
guessed**:
- `resolve_path_dependent_momentum_scores()` already branches on `holding_period < 1` (resamples
  weekly, `period = max(1, round(lookback_period * 4))`), the identical convention every other
  regime-aware scorer uses, no special-casing needed here.
- `multi_timeframe_composite`'s own `resolve_strategy_scores()` dispatch ALWAYS resamples to
  monthly (`scoped_prices.resample("ME").last()`) regardless of `holding_period`. Running it
  under `portfolio2`'s weekly regime does NOT give it a weekly signal, it tests "a
  monthly-timeframe-blended signal, rebalanced weekly", a genuinely different scenario from Epic
  17's "monthly signal, monthly rebalance" run, not a mistake.

**A real, confirmed no-op case, anticipated before running anything, not discovered by
accident**: `portfolio2`'s own `risk_overrides` already sets `use_correlation_penalty: true`
directly (not via a preset), so `correlation_weighted_momentum`'s preset has zero effect vs.
`portfolio2`'s own already-published Epic 16 weekly `momentum` baseline. Confirmed by the real
run: both `dual_momentum` and `correlation_weighted_momentum` came back byte-identical to Epic
16's own baseline (`chosen_lookback=2.0`, holdout Sharpe ~0.52-0.53, CAGR ~3.7%, alpha ~-0.7%),
the same "explicit value already matches, nothing left to differ" shape Epic 17 found for
`dual_momentum` alone (there caused by `use_absolute_momentum` being LIVE-ONLY).

Same cached `crash_test_daily_prices.pkl` proxy panel, no new fetch, same
`pre_registered_split(split_date="2015-01-01")` as every prior epic. `SHY` substitutes for the
default `defensive_ticker` (`"BIL"`, not in the cached panel) for `dual_momentum`/
`absolute_momentum`.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
import dataclasses
from dataclasses import replace

import pandas as pd

from momentum_trading.daily_runner import load_config, apply_strategy_type_preset
from momentum_trading.core import functions_quant_extensions as fnx
from momentum_trading.core.strategy_signals import generate_strategy_monthly_picks
from momentum_trading.backtest.momentum_backtest import BacktestConfig, run_custom_backtest

## 1. Load the real config and the cached proxy-universe price history

In [ ]:
config = load_config()
portfolio2_cfg = config["portfolios_resolved"]["portfolio2"]["cfg"]  # real weekly config

PROXY_TICKERS = [
    "SPY", "QQQ", "DIA", "XLK", "XLF", "XLE", "XLI", "XLP", "XLU", "XLV", "XLY",
    "GLD", "TLT", "IEF", "SHY", "LQD", "IWM",
]

daily_prices = pd.read_pickle("crash_test_daily_prices.pkl")
print(daily_prices.shape, daily_prices.index.min(), "->", daily_prices.index.max())
print(f"strategy_type={portfolio2_cfg.strategy_type} top_n={portfolio2_cfg.top_n} "
      f"holding_period={portfolio2_cfg.holding_period} lookback_period={portfolio2_cfg.lookback_period} "
      f"use_correlation_penalty={portfolio2_cfg.use_correlation_penalty}")

## 2. Pre-registered train / holdout split, and a config builder

In [ ]:
train, holdout = fnx.pre_registered_split(daily_prices, split_date="2015-01-01")
print(f"Train:   {train.index.min().date()} to {train.index.max().date()} ({len(train)} rows)")
print(f"Holdout: {holdout.index.min().date()} to {holdout.index.max().date()} ({len(holdout)} rows)")

LOOKBACK_CANDIDATES = [0.5, 0.75, 1.0, 1.5, 2.0]  # week-quarters -> 2/3/4/6/8 weeks


def build_cfg(strategy_type, overrides=None):
    merged = dataclasses.asdict(portfolio2_cfg)
    merged["strategy_type"] = strategy_type
    if overrides:
        merged.update(overrides)
    merged = apply_strategy_type_preset(merged)
    return BacktestConfig(**merged)


# Matches daily_runner.STRATEGY_TYPE_PRESETS exactly, plus the SHY defensive_ticker
# substitution. NOTE: correlation_weighted_momentum's preset is a KNOWN, EXPECTED no-op here,
# portfolio2's own risk_overrides already sets use_correlation_penalty=True directly.
FULL_SEARCH_VARIANTS = {
    "dual_momentum": {"use_absolute_momentum": True, "use_regime_filter": True, "defensive_ticker": "SHY"},
    "correlation_weighted_momentum": {"use_correlation_penalty": True},
    "rank_sign_momentum": {"sizing_method": "equal_weight"},
    "absolute_momentum": {"defensive_ticker": "SHY"},
    "residual_momentum": {},
    "path_dependent_momentum": {},
}

## 3. Walk-forward + holdout + bootstrap CI, for each of the 6 full-search variants

Same real pipeline as Epic 15-17, one loop over the 6 variants, collecting one combined summary
row each.

In [ ]:
summary_rows = []

for strategy_type, overrides in FULL_SEARCH_VARIANTS.items():
    cfg = build_cfg(strategy_type, overrides)

    wf_results = fnx.run_walk_forward_lookback_search(
        train, PROXY_TICKERS, cfg,
        lookback_candidates=LOOKBACK_CANDIDATES,
        train_years=4, test_years=1, step_years=1,
        metric="Sharpe",
    )
    if wf_results.empty:
        summary_rows.append({"strategy_type": strategy_type, "n_folds": 0})
        continue

    chosen_lookback = float(wf_results["chosen_lookback"].mode().iloc[0])
    mean_train_sharpe = wf_results["train_Sharpe"].mean()
    mean_test_sharpe = wf_results["test_Sharpe"].mean()

    holdout_cfg = replace(cfg, lookback_period=chosen_lookback)
    holdout_start = holdout.index.min()
    picks_full = generate_strategy_monthly_picks(
        daily_prices, PROXY_TICKERS, holdout_cfg, chosen_lookback, holdout_cfg.top_n,
    )
    picks_holdout = picks_full[picks_full.index >= holdout_start]
    holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **holdout_cfg.__dict__)
    ts = holdout_bt.attrs.get("tearsheet", {})

    holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()
    ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)

    summary_rows.append({
        "strategy_type": strategy_type,
        "n_folds": len(wf_results),
        "chosen_lookback": chosen_lookback,
        "mean_train_sharpe": round(mean_train_sharpe, 2),
        "mean_test_sharpe": round(mean_test_sharpe, 2),
        "holdout_sharpe": round(ts.get("Sharpe", float("nan")), 2),
        "holdout_cagr": round(ts.get("CAGR", float("nan")) * 100, 2),
        "holdout_alpha": round(ts.get("Alpha", float("nan")) * 100, 2),
        "ci_low": round(ci["ci_low"], 2),
        "ci_high": round(ci["ci_high"], 2),
    })
    print(f"{strategy_type}: done")

## 4. `multi_timeframe_composite`: holdout-only, always-monthly signal

Its own scoring ALWAYS resamples to monthly regardless of `holding_period` (confirmed by reading
`resolve_strategy_scores()`'s dispatch directly), so this tests a monthly-timeframe-blended
signal rebalanced weekly, not a weekly signal, and no lookback grid search applies.

In [ ]:
mtc_cfg = build_cfg("multi_timeframe_composite")
holdout_start = holdout.index.min()
picks_full = generate_strategy_monthly_picks(
    daily_prices, PROXY_TICKERS, mtc_cfg, mtc_cfg.lookback_period, mtc_cfg.top_n,
)
picks_holdout = picks_full[picks_full.index >= holdout_start]
holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **mtc_cfg.__dict__)
ts = holdout_bt.attrs.get("tearsheet", {})

holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()
ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)

summary_rows.append({
    "strategy_type": "multi_timeframe_composite (holdout-only)",
    "n_folds": None,
    "chosen_lookback": None,
    "mean_train_sharpe": None,
    "mean_test_sharpe": None,
    "holdout_sharpe": round(ts.get("Sharpe", float("nan")), 2),
    "holdout_cagr": round(ts.get("CAGR", float("nan")) * 100, 2),
    "holdout_alpha": round(ts.get("Alpha", float("nan")) * 100, 2),
    "ci_low": round(ci["ci_low"], 2),
    "ci_high": round(ci["ci_high"], 2),
})

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))